# Module 1 — Build an AI agent from scratch

By the end of this notebook you'll have written a working agent: an LLM in a **loop**, with **tools** it can choose to call, that keeps going until your task is done.

We build up in four steps:
1. A single LLM call (one shot)
2. Why the API is *stateless*, and how conversation actually works
3. **Tools** — giving the model actions it can take
4. **The agent loop** — the piece that turns an LLM into an agent

Run each cell in order (Shift+Enter).

## Step 0 — Setup

We load your API key from the `.env` file, then create the Anthropic client.
If the assertion fails, copy `.env.example` to `.env` and paste your key from
https://console.anthropic.com/ .

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
import anthropic

# find_dotenv(usecwd=True) walks up from this notebook's folder to the project root .env
load_dotenv(find_dotenv(usecwd=True))
assert os.environ.get("ANTHROPIC_API_KEY"), \
    "No API key found. Copy .env.example to .env and paste your key, then re-run this cell."

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
#client = anthropic.Client(os.environ.get("ANTHROPIC_API_KEY"))  # alternative way to pass the key

# Opus 4.8 is the most capable model. For lots of cheap practice runs you can switch
# to "claude-haiku-4-5" here — that trade-off is yours to make.
MODEL = "claude-haiku-4-5"
print("Ready. Using model:", MODEL)

Ready. Using model: claude-haiku-4-5


## Step 1 — A single LLM call

Everything starts here: a list of `messages` goes in, one response comes out.
The response `.content` is a **list of blocks**; a plain text answer is one `text` block.

In [4]:
reply = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": "What is the weather in Paris today?"}
    ]
)

In [5]:
print(reply)

Message(id='msg_01PiFSPZYQc5mbFwqE2Do8W4', container=None, content=[TextBlock(citations=None, text='I don\'t have access to real-time information, including current weather data. My knowledge was last updated in April 2024, and I can\'t browse the internet.\n\nTo check today\'s weather in Paris, you can:\n\n- **Google Weather** – Search "Paris weather"\n- **Weather.com** or **Weather.gov**\n- **Local French sources** – Météo-France (the official French meteorological service)\n- **Weather apps** on your phone\n\nThese will give you up-to-date forecasts, temperatures, and conditions.', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=123, output_tokens_details=None, server_t

In [6]:
print(reply.content)

[TextBlock(citations=None, text='I don\'t have access to real-time information, including current weather data. My knowledge was last updated in April 2024, and I can\'t browse the internet.\n\nTo check today\'s weather in Paris, you can:\n\n- **Google Weather** – Search "Paris weather"\n- **Weather.com** or **Weather.gov**\n- **Local French sources** – Météo-France (the official French meteorological service)\n- **Weather apps** on your phone\n\nThese will give you up-to-date forecasts, temperatures, and conditions.', type='text')]


In [7]:
print(reply.content[0])

TextBlock(citations=None, text='I don\'t have access to real-time information, including current weather data. My knowledge was last updated in April 2024, and I can\'t browse the internet.\n\nTo check today\'s weather in Paris, you can:\n\n- **Google Weather** – Search "Paris weather"\n- **Weather.com** or **Weather.gov**\n- **Local French sources** – Météo-France (the official French meteorological service)\n- **Weather apps** on your phone\n\nThese will give you up-to-date forecasts, temperatures, and conditions.', type='text')


In [8]:
print(reply.content[0].text)

I don't have access to real-time information, including current weather data. My knowledge was last updated in April 2024, and I can't browse the internet.

To check today's weather in Paris, you can:

- **Google Weather** – Search "Paris weather"
- **Weather.com** or **Weather.gov**
- **Local French sources** – Météo-France (the official French meteorological service)
- **Weather apps** on your phone

These will give you up-to-date forecasts, temperatures, and conditions.


In [3]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "In one sentence, what is an AI agent?"}],
)

# .content is a list of blocks. For a plain answer, block 0 is a text block.
print(resp.content[0].text)
print("\nstop_reason:", resp.stop_reason)   # "end_turn" = the model finished normally

An AI agent is a software system that perceives its environment, makes decisions based on that information, and takes actions to achieve specific goals.

stop_reason: end_turn


## Step 2 — The API is stateless

The API remembers **nothing** between calls. If you want the model to "remember"
something, you have to send the whole conversation back every time. Let's prove it.

In [3]:
# Turn 1: tell it your name.
r1 = client.messages.create(
    model=MODEL, max_tokens=200,
    messages=[{"role": "user", "content": "My name is Alex. Remember it."}],
)
print("R1:", r1.content[0].text)

# Turn 2 WITHOUT history — a brand new call. It cannot know your name.
r2 = client.messages.create(
    model=MODEL, max_tokens=200,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("\nR2 (no history):", r2.content[0].text)

R1: Got it, Alex! I'll remember your name for our conversation. 

Just so you know, I don't retain memories between separate conversations—so if you start a fresh chat with me later, I won't recall this. But for the duration of our current conversation, I've got you covered.

What can I help you with?

R2 (no history): I don't have access to your name or any personal information about you. I don't retain information between conversations, and I don't have any way to know who you are unless you tell me.

If you'd like, you can share your name and I'll be happy to use it during our conversation!


In [7]:
# Turn 2 WITH history — we replay the whole conversation. Now it knows.
r3 = client.messages.create(
    model=MODEL, max_tokens=200,
    messages=[
        {"role": "user",      "content": "My name is Alex. Remember it."},
        {"role": "assistant", "content": r1.content[0].text},   # the model's own prior reply
        {"role": "user",      "content": "What is my name?"},
    ],
)
print("R3 (with history):", r3.content[0].text)

R3 (with history): Your name is Alex!


**Takeaway:** *you* own the conversation history. Every agent framework you'll ever
use is, underneath, just carefully managing this `messages` list for you.

## Step 3 — Tools: giving the model actions

A tool is just:
1. A normal Python function you write.
2. A JSON **schema** that describes it to the model (name, what it does, its inputs).

The model can't run your code — it can only *ask* to, by returning a `tool_use` block.
*You* run the function and hand back the result. Let's define two tools.

In [8]:
import math

def calculator(expression: str) -> str:
    """Evaluate a math expression like \'2 * (3 + 4)\' or \'sqrt(144)\'."""
    # Restricted namespace: math functions only, no builtins. Fine for a learning
    # sandbox; do NOT eval untrusted input like this in production.
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    try:
        return str(eval(expression, {"__builtins__": {}}, allowed))
    except Exception as e:
        return f"Error: {e}"

def get_weather(city: str) -> str:
    """Return the current weather for a city (mocked for this lesson)."""
    fake = {"paris": "18C, cloudy", "tokyo": "24C, sunny", "new york": "12C, rainy"}
    return fake.get(city.lower(), f"No data for {city}; assume 20C and clear.")

# Map tool name -> function, so the loop can dispatch by name.
TOOL_FUNCS = {"calculator": calculator, "get_weather": get_weather}

# The schemas the model sees. Good descriptions here directly improve tool use.
TOOLS = [
    {
        "name": "calculator",
        "description": "Evaluate a math expression. Use for any arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "e.g. \'2 * (3 + 4)\'"}
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. \'Paris\'"}
            },
            "required": ["city"],
        },
    },
]
print("Defined tools:", list(TOOL_FUNCS))

Defined tools: ['calculator', 'get_weather']


### See a tool call happen (one round)

We pass `tools=TOOLS`. When the model wants a tool, it stops with
`stop_reason == "tool_use"` and its `.content` contains a `tool_use` block with the
tool `name`, an `id`, and the `input` args it chose. Notice: **it does not answer yet** —
it's waiting for us to run the tool.

In [9]:
resp = client.messages.create(
    model=MODEL, max_tokens=1024, tools=TOOLS,
    messages=[{"role": "user", "content": "What is 15% of 240?"}],
)

print("stop_reason:", resp.stop_reason, "\n")
for block in resp.content:
    if block.type == "text":
        print("TEXT:", block.text)
    elif block.type == "tool_use":
        print("TOOL_USE:", block.name, "id=", block.id)
        print("   input:", block.input)

stop_reason: tool_use 

TEXT: I'll calculate that for you.
TOOL_USE: calculator id= toolu_01S5FsUjp9ozZXHdjKL5bgiu
   input: {'expression': '0.15 * 240'}


## Step 4 — The agent loop

Now the key move. A single call can only *request* a tool. To finish the task we:

1. Call the model.
2. If it stopped to use tools → run each tool, append the results, **call again**.
3. Repeat until it stops with something other than `tool_use` (it's ready to answer).

That `while` loop is what makes it an *agent*. Two rules the API enforces:
- Append the assistant's **entire** `.content` back (so the model sees its own tool request).
- Reply with a `tool_result` for **every** `tool_use`, matched by `tool_use_id`, in a single user message.

In [10]:
def run_agent(user_message, max_turns=10, verbose=True):
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL, max_tokens=2048, tools=TOOLS, messages=messages,
        )
        # Always append the assistant turn (text + any tool_use blocks) to history.
        messages.append({"role": "assistant", "content": resp.content})

        # No tool requested -> the model is answering. We're done.
        if resp.stop_reason != "tool_use":
            return "".join(b.text for b in resp.content if b.type == "text")

        # Otherwise: run every requested tool and collect the results.
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"[turn {turn}] -> {block.name}({block.input})")
                result = TOOL_FUNCS[block.name](**block.input)
                if verbose:
                    print(f"           result: {result}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,     # must match the tool_use it answers
                    "content": str(result),
                })

        # Feed all results back as one user message; loop continues.
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: reached max_turns without a final answer."

print("run_agent defined.")

run_agent defined.


### Run it on a multi-step task

This question needs **two** tools and some reasoning in between — watch the loop take
several turns before it answers.

In [11]:
answer = run_agent(
    "What's the weather in Tokyo, and what is 24 degrees Celsius in Fahrenheit? "
    "Use the calculator for the conversion."
)
print("\n=== FINAL ANSWER ===")
print(answer)

[turn 0] -> get_weather({'city': 'Tokyo'})
           result: 24C, sunny
[turn 0] -> calculator({'expression': '24 * 9/5 + 32'})
           result: 75.2

=== FINAL ANSWER ===
Here's what I found:

- **Weather in Tokyo:** 24°C and sunny ☀️
- **Temperature conversion:** 24°C = **75.2°F**

As a nice coincidence, Tokyo's current temperature is exactly the 24°C you asked to convert!


## Recap

You built an agent. The whole thing is:

- **messages** — the conversation history you own and replay each call.
- **tools** — functions + schemas the model can *choose* to call.
- **the loop** — run tools, feed results back, repeat until the model answers.

Everything more advanced (memory, planning, multi-agent, frameworks) is a variation on
this core loop.

### Exercises (try before Module 2)
1. Add a third tool — e.g. `get_time()` (use `datetime`) — with its schema, and ask a
   question that needs it.
2. Set `verbose=True` and trace *why* the loop runs the number of turns it does.
3. Break a tool on purpose (raise an exception) and watch how the model reacts to the
   error string you return. What happens if you set `"is_error": True` on the tool_result?
4. Ask something needing **no** tools. How many turns does the loop take, and why?

### Next: Module 2 — real tools & robustness
Multiple tools, proper error handling, parallel tool calls, and forcing tool choice.
When you're ready, ask me to build `02_real_tools.ipynb`.